# Week 62, Day 5: Weak references

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w062d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import weakref

class Photo:
    def __init__(self, name):
        self.name = name

p = Photo("beach")
r = weakref.ref(p)
print(r() is p, r().name)
del p
print(r())

cache = weakref.WeakValueDictionary()
keep = Photo("hills")
cache["hills"] = keep
cache["sea"] = Photo("sea")
print(sorted(cache.keys()))

In [ ]:
import weakref

class Session:
    def __init__(self, user):
        self.user = user

events = []
s = Session("ada")
r = weakref.ref(s, lambda ref: events.append("ada's session ended"))

target = r()
if target is not None:
    print("still here:", target.user)
del target                                  # our temporary strong reference

print(weakref.getweakrefcount(s), r() is s)
del s
print(r(), events)

In [ ]:
import weakref

def try_ref(obj):
    try:
        weakref.ref(obj)
        return "yes"
    except TypeError:
        return "no"

class Plain:
    pass

class Slotted:
    __slots__ = ("x",)

class SlottedWeak:
    __slots__ = ("x", "__weakref__")

class MyList(list):
    pass

for thing in (Plain(), Slotted(), SlottedWeak(), MyList(), [], {}, 42, "s", (1,), {1}, try_ref):
    print(type(thing).__name__, try_ref(thing))

In [ ]:
import weakref

class Texture:
    loads = 0
    def __init__(self, name):
        Texture.loads += 1
        self.name = name

cache = weakref.WeakValueDictionary()

def get_texture(name):
    tex = cache.get(name)
    if tex is None:
        tex = Texture(name)
        cache[name] = tex
    return tex

grass = get_texture("grass")
again = get_texture("grass")
print(grass is again, Texture.loads)
del grass, again
get_texture("grass")                        # not kept, so loaded again (and dropped again)
print(Texture.loads, len(cache))

notes = weakref.WeakKeyDictionary()
class Window:
    pass
w = Window()
notes[w] = "main window"
print(len(notes))
del w
print(len(notes))

In [ ]:
import weakref

class Panel:
    def __init__(self, name):
        self.name = name
    def refresh(self, data):
        return f"{self.name} shows {data}"

panel = Panel("sales")
plain = weakref.ref(panel.refresh)
print(plain())                              # the temporary bound method is already gone

method = weakref.WeakMethod(panel.refresh)
handler = method()
print(handler("42"))
del handler, panel
print(method())

In [ ]:
import weakref

log = []

class TempDir:
    def __init__(self, path):
        self.path = path
        self._finalizer = weakref.finalize(self, log.append, f"removed {path}")

    def close(self):
        self._finalizer()                    # run now; never again

    @property
    def closed(self):
        return not self._finalizer.alive

a = TempDir("/tmp/a")
a.close()
a.close()
print(a.closed, log)

b = TempDir("/tmp/b")
del b                                        # forgot to close: the safety net runs
print(log)

> **Key idea.** weakref.ref(obj) refers to obj without keeping it alive: call it to get the object or None, keep the result in a name while using it, and pass a callback to hear when the object dies. Instances of your classes, functions, sets and classes can be weakly referenced; int, str, tuple, list and dict can't, and __slots__ classes need "__weakref__". WeakValueDictionary makes caches and registries that forget dead objects, WeakKeyDictionary attaches data to objects without keeping them alive, and WeakSet tracks live members. Bound methods are temporary objects, so store observers with WeakMethod. weakref.finalize(obj, func, *args) runs cleanup once, at collection or exit, and can be called early; its function and arguments must not refer to obj.

### Exercise: Live or dead

Write `can_weakref(obj)`: True if a weak reference to `obj` can be made. Then `live_targets(refs)`: given a list of weak references, the list of objects still alive (in order), and `track(objs, log)`: return a list of weak references to the objects, each with a callback that appends the **position** of that object in `objs` to `log` when it dies.

In [ ]:
import weakref

def can_weakref(obj):
    ___

def live_targets(refs):
    ___

def track(objs, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_refs():\n    \"weakref.ref\"\n    class Item:\n        pass\n    assert can_weakref(Item()) and can_weakref({1}) and can_weakref(can_weakref)\n    assert not can_weakref([]) and not can_weakref(5) and not can_weakref(\"x\")\n    items = [Item(), Item(), Item()]\n    log = []\n    refs = track(items, log)\n    assert live_targets(refs) == items\n    middle = items.pop(1)\n    assert log == []\n    del middle\n    assert log == [1] and live_targets(refs) == items\n    items.clear()\n    assert sorted(log) == [0, 1, 2] and live_targets(refs) == []")

### Exercise: A cache that lets go

Write `ModelCache(loader)`: `get(name)` returns the live object for that name if there is one, otherwise calls `loader(name)`, remembers the result **weakly** and returns it. `loads` counts how many times `loader` was called, `cached()` is the sorted list of names whose objects are still alive, and `forget(name)` drops an entry (no error if it's absent). Use `weakref.WeakValueDictionary`.

In [ ]:
import weakref

class ModelCache:
    def __init__(self, loader):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"WeakValueDictionary\"\n    class Model:\n        def __init__(self, name):\n            self.name = name\n    c = ModelCache(Model)\n    a = c.get(\"tree\")\n    assert c.get(\"tree\") is a and c.loads == 1 and c.cached() == [\"tree\"]\n    b = c.get(\"rock\")\n    assert c.cached() == [\"rock\", \"tree\"]\n    del a\n    assert c.cached() == [\"rock\"]\n    a2 = c.get(\"tree\")\n    assert c.loads == 3 and a2.name == \"tree\"\n    c.forget(\"rock\")\n    c.forget(\"nothing\")\n    assert c.get(\"rock\") is not b and c.loads == 4")

### Exercise: Notes on other people's objects

Write `Labels`, a side table that attaches a set of labels to objects without keeping them alive: `add(obj, label)`, `labels(obj)` (a sorted list; empty if none), `remove(obj, label)` (no error if absent), `objects_with(label)` (a list of the live objects carrying that label, in no particular order), and `len(labels)` (how many live objects have any labels). Use `weakref.WeakKeyDictionary`.

In [ ]:
import weakref

class Labels:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_side():\n    \"WeakKeyDictionary\"\n    class Doc:\n        pass\n    t = Labels()\n    a, b = Doc(), Doc()\n    t.add(a, \"draft\")\n    t.add(a, \"urgent\")\n    t.add(b, \"draft\")\n    assert t.labels(a) == [\"draft\", \"urgent\"] and len(t) == 2\n    assert set(map(id, t.objects_with(\"draft\"))) == {id(a), id(b)}\n    t.remove(a, \"urgent\")\n    t.remove(a, \"none\")\n    assert t.labels(a) == [\"draft\"] and t.labels(Doc()) == []\n    del b\n    assert len(t) == 1 and t.objects_with(\"draft\") == [a]")

### Exercise: Observers that can leave

Write `Signal()`: `connect(method)` stores a **bound method** weakly using `weakref.WeakMethod`; `emit(*args)` calls every live handler in the order they were connected and returns a list of their results; dead handlers are dropped. `count()` is the number of handlers whose objects are still alive. Connecting the same bound method twice keeps only one (compare the methods they produce).

In [ ]:
import weakref

class Signal:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_signal():\n    \"WeakMethod\"\n    class Screen:\n        def __init__(self, name):\n            self.name = name\n        def show(self, value):\n            return f\"{self.name}:{value}\"\n    sig = Signal()\n    a, b = Screen(\"a\"), Screen(\"b\")\n    sig.connect(a.show)\n    sig.connect(b.show)\n    sig.connect(a.show)\n    assert sig.count() == 2 and sig.emit(1) == [\"a:1\", \"b:1\"]\n    del a\n    assert sig.count() == 1 and sig.emit(2) == [\"b:2\"]\n    del b\n    assert sig.emit(3) == [] and sig.count() == 0")

### Exercise: A safety net that runs once

Write `Lease(name, log)`: on creation it appends `"open " + name` to the list `log` and registers a `weakref.finalize` that appends `"release " + name`. `release()` runs the cleanup early (only the first call does anything), `active` is True until the cleanup has run, and an unreleased lease that is garbage collected is released automatically. The finaliser must not refer to the lease itself.

In [ ]:
import weakref

class Lease:
    def __init__(self, name, log):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_finalize():\n    \"weakref.finalize\"\n    log = []\n    a = Lease(\"a\", log)\n    assert a.active and log == [\"open a\"]\n    a.release()\n    a.release()\n    assert not a.active and log == [\"open a\", \"release a\"]\n    b = Lease(\"b\", log)\n    del b\n    assert log[-1] == \"release b\"\n    c = Lease(\"c\", log)\n    import gc\n    c.me = c                               # a cycle: freed by the collector\n    del c\n    gc.collect()\n    assert log[-1] == \"release c\" and log.count(\"release c\") == 1")

### Bug hunt: Cached and gone

The thumbnail is put straight into a `WeakValueDictionary`, and nothing else refers to it, so it dies at once and this prints `missing`. Keep a strong reference to it in a name while it's in use so it prints `found`.

In [ ]:
import weakref

class Thumb:
    pass

cache = weakref.WeakValueDictionary()
cache["home"] = Thumb()
print("found" if "home" in cache else "missing")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a strong reference\"\n    expect_output(\"found\")")

### Bug hunt: A method that dies at once

`weakref.ref(chart.redraw)` refers to a temporary bound method that dies straight away, even though `chart` is alive, so this prints `dead`. Use `weakref.WeakMethod` so it prints `alive`.

In [ ]:
import weakref

class Chart:
    def redraw(self):
        return "drawn"

chart = Chart()
handler = weakref.ref(chart.redraw)
print("alive" if handler() is not None else "dead")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"WeakMethod\"\n    expect_output(\"alive\")")

### Bug hunt: A finaliser that keeps its object alive

The finaliser is given `self.cleanup`, a bound method, which refers to the object, so the object can never die and the cleanup never runs: this prints `[]`. Register a function that doesn't refer to the object, passing it the name, so it prints `['r1']`.

In [ ]:
import weakref

log = []

class Resource:
    def __init__(self, name):
        self.name = name
        weakref.finalize(self, self.cleanup)

    def cleanup(self):
        log.append(self.name)

r = Resource("r1")
del r
print(log)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no reference to self\"\n    expect_output(\"['r1']\")")

### Puzzle: A weak cache lookup

Arrange the lines of a function that returns a cached object if it's alive, loading it otherwise. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what r() returns after the object it refers to has died
q = ___   # True or False: weakref.ref([1, 2]) works
r = ___   # True or False: a WeakKeyDictionary keeps its keys alive
s = ___   # True or False: weakref.ref(obj.method)() is None straight away, while obj is alive
t = ___   # how many times a weakref.finalize cleanup can run at most

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is None and q is False and r is False, \"p, q and r\"\n    assert s is True and t == 1, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a weak event bus

Write `EventBus()`, which holds subscribers without keeping them alive:

- `subscribe(event, handler)`: a handler is either a **bound method** (store it with `WeakMethod`, with a callback that removes the subscription the moment its object dies) or a plain **function** (held strongly). Subscribing the same handler to the same event twice does nothing. Returns a function that unsubscribes it when called.
- `publish(event, *args)`: call the live handlers for that event in subscription order and return the number called. A handler may unsubscribe itself, or subscribe new handlers, while being called: those changes take effect from the next `publish`.
- `count(event)`: the number of subscriptions to that event. Because of the callbacks, this drops **as soon as** a subscriber dies, before any publish.
- `events()`: the sorted list of events with at least one subscription.

In [ ]:
import weakref

class EventBus:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bus():\n    \"a weak event bus\"\n    bus = EventBus()\n    seen = []\n    class Viewer:\n        def __init__(self, name):\n            self.name = name\n        def on_save(self, doc):\n            seen.append((self.name, doc))\n    def audit(doc):\n        seen.append((\"audit\", doc))\n    a, b = Viewer(\"a\"), Viewer(\"b\")\n    bus.subscribe(\"save\", a.on_save)\n    bus.subscribe(\"save\", audit)\n    stop_b = bus.subscribe(\"save\", b.on_save)\n    bus.subscribe(\"save\", a.on_save)\n    assert bus.count(\"save\") == 3 and bus.publish(\"save\", \"d1\") == 3\n    assert seen == [(\"a\", \"d1\"), (\"audit\", \"d1\"), (\"b\", \"d1\")]\n    del a\n    assert bus.count(\"save\") == 2\n    stop_b()\n    stop_b()\n    assert bus.publish(\"save\", \"d2\") == 1 and seen[-1] == (\"audit\", \"d2\")\n    assert bus.events() == [\"save\"] and bus.publish(\"nothing\") == 0 and bus.count(\"nothing\") == 0\n\ndef test_changes_while_publishing():\n    \"changes during publish\"\n    bus = EventBus()\n    calls = []\n    def once(x):\n        calls.append(\"once\")\n        unsub()\n    def adder(x):\n        calls.append(\"adder\")\n        bus.subscribe(\"tick\", late)\n    def late(x):\n        calls.append(\"late\")\n    unsub = bus.subscribe(\"tick\", once)\n    bus.subscribe(\"tick\", adder)\n    assert bus.publish(\"tick\", 1) == 2 and calls == [\"once\", \"adder\"]\n    calls.clear()\n    assert bus.publish(\"tick\", 2) == 2 and calls == [\"adder\", \"late\"]\n    class Temp:\n        def go(self, x):\n            pass\n    t = Temp()\n    bus.subscribe(\"other\", t.go)\n    assert bus.events() == [\"other\", \"tick\"]\n    del t\n    assert bus.events() == [\"tick\"]")

## Warm-up

A weak reference gives the object back while it's alive and `None` once it's gone. The cache entry for `"sea"` vanished at once, because nothing **strong** referred to that photo.

## weakref.ref

`r = weakref.ref(obj)` makes a weak reference. **Calling** it, `r()`, returns the object if it's still alive and `None` if it has been freed. Since the object can disappear between two lines, always call once and keep the result in a name while you use it:

```
target = r()
if target is not None:
    target.refresh()
```

`weakref.ref(obj, callback)` also registers a **callback**, called with the weak reference itself just after the object dies: the place to remove a dead entry from your own data structure. Weak references to the same object without callbacks are often the same reference object, and they're hashable if the object is, so they can be dict keys.

Remember yesterday's mechanics: in CPython an object without cycles dies the moment its last strong reference goes, so the callback runs at once; an object in a cycle dies only when the cycle collector runs.

## What can be weakly referenced

An object can be weakly referenced only if its type makes room for a list of weak references. That includes instances of your own classes, functions, methods' functions, sets, frozensets, classes and modules. It **excludes** several built-ins: `int`, `str`, `bytes`, `tuple`, `list` and `dict` (a subclass of `list` or `dict` works, though).

A class with `__slots__` (Week 31) has no room unless you add it: include `"__weakref__"` in the slots.

Weak references make sense for **large, identity-carrying** objects (windows, documents, connections, cache entries), not for small values: a weak reference to a number would make no sense, since equal numbers are interchangeable.

## Weak containers

The `weakref` module has ready-made containers that clean themselves up:

- **`WeakValueDictionary`**: values are held weakly; an entry disappears when its value dies. Ideal for **caches** and **registries** ("give me the live object with this id, if there is one") and for interning: never two live objects for the same key.
- **`WeakKeyDictionary`**: keys are held weakly; an entry disappears when its key dies. Ideal for **side tables**: attaching extra data to objects you don't own (and can't add attributes to) without keeping them alive.
- **`WeakSet`**: a set of weakly held members, ideal for "everything currently registered" (open windows, live connections).

Two cautions: iterating over a weak container while objects are dying can change it, so take a `list(...)` of it first if you'll do anything that frees objects; and the number of entries is only meaningful once dead objects have actually been freed (in CPython, at once without cycles).

## Bound methods and WeakMethod

There's a trap with observers. `obj.method` creates a **new bound method object** every time you write it, and that object is what you'd be referring to. So `weakref.ref(obj.method)` dies **immediately**, even though `obj` is alive: the temporary bound method had no other references. And holding the bound method **strongly** keeps `obj` alive (yesterday's observer leak).

`weakref.WeakMethod(obj.method)` solves this: it holds the **object** and the **function** weakly and rebuilds the bound method when you call it, returning `None` once the object has gone. Use it whenever you store callbacks that are methods of objects that should be free to die. (Plain functions are usually module-level and live forever, so holding them strongly is fine.)

## weakref.finalize

`weakref.finalize(obj, func, *args, **kwargs)` registers `func(*args, **kwargs)` to run **when `obj` is garbage collected**, or when the program exits, whichever comes first. It returns a finaliser object you can also **call** yourself to run the cleanup early; either way it runs **at most once**, and `finaliser.alive` tells you whether it's still pending.

It's a better safety net than `__del__` (Week 61):

- it's guaranteed to run at most once, and at exit if not before;
- it doesn't let the object "resurrect" itself, and cycles containing the object can be collected normally;
- the cleanup is separate from the object, so a `close()` method and the safety net share one code path.

The one rule: `func` and `args` **must not refer to the object itself** (no `self.close`, no `self`), or the finaliser keeps the object alive forever. Pass the pieces the cleanup needs, such as a file handle or a path, instead.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a long-running program you use (an editor, a browser): which of its parts would you expect to use weak references? Tomorrow is the Forge: a memory-aware object store that brings the whole week together.